# ICML Ablation — AetherLM on Kaggle TPU v5e-8

Runs ablation experiments sequentially on a single Kaggle TPU.
Each experiment trains from scratch with Chinchilla-optimal tokens.

**Kaggle secrets required:** `WANDB_API_KEY`, `WANDB_ENTITY`, `WANDB_PROJECT`, `GITHUB_TOKEN`

In [3]:
# ── Install dependencies ────────────────────────────────────────────
!pip install -q lm-eval wandb duckdb
!pip install -U git+https://github.com/azettaai/nmn.git -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
# ── Kaggle secrets ──────────────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_ENTITY"] = user_secrets.get_secret("WANDB_ENTITY")
os.environ["WANDB_PROJECT"] = user_secrets.get_secret("WANDB_PROJECT")

In [10]:
# ── Clone repo + install ────────────────────────────────────────────
!git clone -b scale https://{GITHUB_TOKEN}@github.com/azettaai/aetherlm.git
%cd aetherlm
!pip install -e . -q

Cloning into 'aetherlm'...
remote: Enumerating objects: 3115, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 3115 (delta 56), reused 52 (delta 41), pack-reused 3005 (from 2)
Receiving objects: 100% (3115/3115), 2.15 MiB | 14.45 MiB/s, done.
Resolving deltas: 100% (2179/2179), done.
/kaggle/working/aetherlm

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
# ── Environment check ───────────────────────────────────────────────
import jax
import jax.numpy as jnp

n_devices = jax.device_count()
platform = jax.devices()[0].platform
print(f"JAX {jax.__version__}  |  {n_devices} devices  |  {platform}")
for d in jax.devices():
    print(f"  {d}")

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1774814400.146464      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


JAX 0.9.2  |  8 devices  |  tpu
  TPU_0(process=0,(0,0,0,0))
  TPU_1(process=0,(1,0,0,0))
  TPU_2(process=0,(0,1,0,0))
  TPU_3(process=0,(1,1,0,0))
  TPU_4(process=0,(0,2,0,0))
  TPU_5(process=0,(1,2,0,0))
  TPU_6(process=0,(0,3,0,0))
  TPU_7(process=0,(1,3,0,0))


In [24]:
# ══════════════════════════════════════════════════════════════════════
# Ablation configuration
# ══════════════════════════════════════════════════════════════════════

import datetime as dt

# ── Shared hyperparameters (same across all arms) ───────────────────
EMBED_DIM       = 768
NUM_HEADS       = 12
NUM_LAYERS      = 12
FF_DIM          = 3072
MAXLEN          = 1024
VOCAB_SIZE      = 32000
TOKENIZER       = "mistralai/Mistral-7B-v0.1"
TIE_EMBEDDINGS  = True
USE_ROPE        = False
USE_REMAT       = True
REMAT_POLICY    = "dots_saveable"
DROPOUT         = 0.0
PARAM_DTYPE     = "bfloat16"
ATTN_KERNEL     = "auto"

LR              = 3e-3
WARMUP          = 2000
END_LR          = 1e-5
OPT_TYPE        = "lamb"
WEIGHT_DECAY    = 0.01
BETA1           = 0.9
BETA2           = 0.999
EPS             = 1e-6
LABEL_SMOOTH    = 0.0
Z_LOSS          = 1e-4
MAX_GRAD_NORM   = 0.0
GRAD_DTYPE      = "bfloat16"

TRAIN_DATASET       = "HuggingFaceFW/fineweb-edu"
TRAIN_DATASET_CFG   = "sample-100BT"
VAL_DATASET         = "wikitext"
VAL_DATASET_CFG     = "wikitext-103-v1"
VAL_DATASET_SPLIT   = "validation"

# ── Per-device batch & Chinchilla scaling ───────────────────────────
BATCH_PER_DEVICE = 32   # conservative for v5e-8 (16GB HBM/chip)
GRAD_ACCUM       = 1
GLOBAL_BATCH     = BATCH_PER_DEVICE * n_devices * GRAD_ACCUM
TOK_PER_STEP     = GLOBAL_BATCH * MAXLEN

# Chinchilla-optimal tokens for 124M params
CHINCHILLA_TOKENS = 2_621_440_000
STEPS = max(CHINCHILLA_TOKENS // TOK_PER_STEP, 1)

print(f"Global batch : {GLOBAL_BATCH}  ({TOK_PER_STEP:,} tok/step)")
print(f"Steps        : {STEPS:,}  ({STEPS * TOK_PER_STEP / 1e9:.1f}B tokens)")

Global batch : 256  (262,144 tok/step)
Steps        : 5,000  (1.3B tokens)


In [25]:
# ══════════════════════════════════════════════════════════════════════
# Experiment arms
# ══════════════════════════════════════════════════════════════════════
#
# Edit EXPERIMENTS to choose which arms to run in this session.
# Each run takes ~30-120 min depending on device count and steps.
#
# Full Batch A has 15 arms x 3 seeds = 45 runs.
# Pick a subset that fits in Kaggle's 9h session limit.
EXPERIMENTS = [
# (name, model_type, norm_type, mlp_type, seed, extra_overrides)
# ── Batch A: Factorized Ablation (15 arms) ──
#     ("A1_baseline_s0",           "gpt",                              "pre_rmsnorm",
# "gelu",    0, {}),
#     ("A2_yat_mlp_s0",            "gpt_yatnmn_pre_rmsnorm",           "pre_rmsnorm",
# "yatnmn",  0, {}),
      ("A3_yat_attn_s0",           "gpt_yat_attn_pre_rmsnorm",         "pre_rmsnorm",
  "gelu",    0, {}),
#     ("A4_no_norm_s0",            "gpt",                              "none",
  # "gelu",    0, {}),
  #     ("A5_yat_both_s0",           "gpt_yatnmn_yat",                   "pre_rmsnorm",
  # "yatnmn",  0, {}),
  #     ("A6_yat_mlp_nf_s0",         "gpt_yatnmn",                       "none",
  # "yatnmn",  0, {}),
  #     ("A7_yat_attn_nf_s0",        "gpt_yat_attn",                     "none",
  # "gelu",    0, {}),
      ("A8_full_aether_s0",        "gpt_yatnmn_yat",                   "norm_free",
  "yatnmn",  0, {}),
  #     ("A9_yat_performer_s0",      "gpt_yatnmn_yat_performer_v2",      "norm_free",
  # "yatnmn",  0, {"num_anchor_features": 8, "num_prf_features": 8}),
  #     ("A10_gpt2_baseline_s0",     "gpt",
  # "post_layernorm","gelu",    0, {}),
  #     ("A11_aether_post_ln_s0",    "gpt_yatnmn_yat",
  # "post_layernorm","yatnmn",  0, {}),
      ("A12_yat_attn_perf_s0",     "gpt_yat_attn_performer_v2",        "pre_rmsnorm",
  "gelu",    0, {"num_anchor_features": 8, "num_prf_features": 8}),
      ("A13_performer_s0",         "gpt_performer",                    "pre_rmsnorm",
  "gelu",    0, {}),
      ("A14_cosformer_s0",         "gpt_cosformer",                    "pre_rmsnorm",
  "gelu",    0, {}),
  #     ("A15_aether_b128_s0",       "gpt_yatnmn_yat",                   "norm_free",
  # "yatnmn",  0, {}),  # batch_override=128 handled separately
  ]

print(f"{len(EXPERIMENTS)} experiments queued:")
for name, mt, nt, mlp, seed, _ in EXPERIMENTS:
    print(f"  {name:<25s}  model={mt:<30s}  norm={nt:<14s}  mlp={mlp}  seed={seed}")

5 experiments queued:
  A3_yat_attn_s0             model=gpt_yat_attn_pre_rmsnorm        norm=pre_rmsnorm     mlp=gelu  seed=0
  A8_full_aether_s0          model=gpt_yatnmn_yat                  norm=norm_free       mlp=yatnmn  seed=0
  A12_yat_attn_perf_s0       model=gpt_yat_attn_performer_v2       norm=pre_rmsnorm     mlp=gelu  seed=0
  A13_performer_s0           model=gpt_performer                   norm=pre_rmsnorm     mlp=gelu  seed=0
  A14_cosformer_s0           model=gpt_cosformer                   norm=pre_rmsnorm     mlp=gelu  seed=0


In [26]:
# ══════════════════════════════════════════════════════════════════════
# Config builder
# ══════════════════════════════════════════════════════════════════════

from aetherlm.core.config import AetherConfig


def build_experiment_config(exp_name, model_type, norm_type, mlp_type, seed, extra):
    ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"icml_{exp_name}_kaggle_{ts}"

    return AetherConfig.from_dict({
        "training": {
            "mode": "causal",
            "batch_size": BATCH_PER_DEVICE,
            "learning_rate": LR,
            "max_steps": STEPS,
            "grad_accumulation_steps": GRAD_ACCUM,
            "label_smoothing": LABEL_SMOOTH,
            "z_loss_coeff": Z_LOSS,
            "grad_dtype": GRAD_DTYPE,
            "max_inflight_steps": 4,
            "ce_chunk_size": 256,
            "log_grad_norm": True,
            "log_param_norm": True,
        },
        "model": {
            "model_type": model_type,
            "embed_dim": EMBED_DIM,
            "num_heads": NUM_HEADS,
            "num_layers": NUM_LAYERS,
            "ff_dim": FF_DIM,
            "maxlen": MAXLEN,
            "vocab_size": VOCAB_SIZE,
            "tokenizer_name": TOKENIZER,
            "tie_embeddings": TIE_EMBEDDINGS,
            "dropout_rate": DROPOUT,
            "param_dtype": PARAM_DTYPE,
            "norm_type": norm_type,
            "mlp_type": mlp_type,
            "fused_yatnmn": True,
            "use_rope": USE_ROPE,
            "use_remat": USE_REMAT,
            "remat_policy": REMAT_POLICY,
            "attention_kernel": ATTN_KERNEL,
            "extra": {
                "num_recurrences": 1,
                "yatnmn_epsilon": 0.001,
                "yat_epsilon": 1.0,
                "yat_alpha": "learnable",
                "yat_normalization": "l1",
                "use_rope": True,
                "seed": seed,
                **extra,
            },
        },
        "optimizer": {
            "type": OPT_TYPE,
            "warmup_steps": WARMUP,
            "end_lr": END_LR,
            "weight_decay": WEIGHT_DECAY,
            "max_grad_norm": MAX_GRAD_NORM,
            "adam_beta1": BETA1,
            "adam_beta2": BETA2,
            "adam_eps": EPS,
            "schedule_type": "warmup_cosine",
        },
        "data": {
            "train_dataset": TRAIN_DATASET,
            "train_dataset_config": TRAIN_DATASET_CFG,
            "train_streaming": True,
            "packing": True,
            "tokenize_num_workers": 8,
            "prefetch_buffer": 32,
            "num_prefetch_workers": 2,
            "val_dataset": VAL_DATASET,
            "val_dataset_config": VAL_DATASET_CFG,
            "val_dataset_split": VAL_DATASET_SPLIT,
        },
        "checkpoint": {
            "dir": f"/kaggle/working/checkpoints/{run_name}",
            "interval": 1000,
            "zip_on_save": True,
        },
        "logging": {
            "run_name": run_name,
            "use_wandb": bool(os.environ.get("WANDB_API_KEY")),
            "wandb_entity": os.environ.get("WANDB_ENTITY", ""),
            "wandb_project": os.environ.get("WANDB_PROJECT", "icmlDelulu"),
            "log_interval": 20,
            "print_interval": 20,
            "eval_interval": STEPS,
            "eval_steps": 50,
            "watch_weights": True,
            "weight_watch_interval": 100,
            "log_weight_histograms": True,
            "enable_duckdb": True,
            "duckdb_path": f"{run_name}.duckdb",
            "duckdb_flush_every": 10,
            "interpret_interval": 500,
            "interpret_anomaly": True,
        },
        "tpu": {
            "precision": "bf16",
        },
    })

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Run all experiments sequentially
# ══════════════════════════════════════════════════════════════════════

import gc
import time
import json
import traceback

from aetherlm.core.engine import AetherEngine
from aetherlm.data.causal import load_causal_datasets

results = []

for i, (exp_name, model_type, norm_type, mlp_type, seed, extra) in enumerate(EXPERIMENTS):
    print()
    print("=" * 70)
    print(f"[{i+1}/{len(EXPERIMENTS)}] {exp_name}")
    print(f"  model={model_type}  norm={norm_type}  mlp={mlp_type}  seed={seed}")
    print("=" * 70)

    t0 = time.time()
    status = "failed"
    final_loss = None
    final_val_loss = None

    try:
        # Build config
        config = build_experiment_config(exp_name, model_type, norm_type, mlp_type, seed, extra)

        # Initialize engine (builds model, optimizer, mesh)
        engine = AetherEngine(config)
        engine.setup()

        # Load data
        dcfg = config.data
        train_ds, val_ds = load_causal_datasets(
            maxlen=MAXLEN,
            tokenizer_name=TOKENIZER,
            train_dataset_name=dcfg.train_dataset,
            train_dataset_config=dcfg.train_dataset_config,
            train_streaming=dcfg.train_streaming,
            val_dataset_name=dcfg.val_dataset,
            val_dataset_config=dcfg.val_dataset_config,
            val_dataset_split=dcfg.val_dataset_split,
        )

        # Train
        engine.train(train_data=train_ds, val_data=val_ds, num_steps=STEPS)

        stats = engine.metrics.compute_statistics()
        final_loss = stats.get("best_train_loss")
        final_val_loss = stats.get("best_val_loss")
        status = "done"

        print(f"\n  DONE  loss={final_loss}  val_loss={final_val_loss}")

    except Exception as e:
        print(f"\n  FAILED: {e}")
        traceback.print_exc()

    elapsed = time.time() - t0
    results.append({
        "experiment": exp_name,
        "model_type": model_type,
        "norm_type": norm_type,
        "mlp_type": mlp_type,
        "seed": seed,
        "status": status,
        "train_loss": final_loss,
        "val_loss": final_val_loss,
        "elapsed_min": round(elapsed / 60, 1),
        "steps": STEPS,
        "global_batch": GLOBAL_BATCH,
    })

    # Save incremental results
    with open("/kaggle/working/ablation_results.json", "w") as f:
        json.dump(results, f, indent=2)

    # Cleanup to free HBM for next experiment
    try:
        del engine, train_ds, val_ds, config
    except NameError:
        pass
    gc.collect()
    print(f"  Cleanup done. Elapsed: {elapsed/60:.1f} min")

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 70)


[1/5] A3_yat_attn_s0
  model=gpt_yat_attn_pre_rmsnorm  norm=pre_rmsnorm  mlp=gelu  seed=0


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

[engine] [worker 0] step 0 batch shapes: {'input_ids': 'list[256]', 'labels': 'list[256]', 'attention_mask': 'list[256]'}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════

import json

with open("/kaggle/working/ablation_results.json") as f:
    results = json.load(f)

print(f"{'Experiment':<28s} {'Status':>8s} {'Loss':>10s} {'Val Loss':>10s} {'Time':>8s}")
print("-" * 70)
for r in results:
    loss_str = f"{r['train_loss']:.4f}" if r['train_loss'] else "---"
    val_str = f"{r['val_loss']:.4f}" if r['val_loss'] else "---"
    print(f"{r['experiment']:<28s} {r['status']:>8s} {loss_str:>10s} {val_str:>10s} {r['elapsed_min']:>6.1f}m")

n_ok = sum(1 for r in results if r['status'] == 'done')
print(f"\n{n_ok}/{len(results)} succeeded")